# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
[https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

This notebook strictly references all entities (record sets, fields, columns) by their `@id`.

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlcimport pandas as pdimport json
# Define the dataset URLcroissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'
# Load the dataset metadata using mlcroissantdataset = mlc.Dataset(croissant_url)metadata = dataset.metadata
# Print basic metadataprint(f"{metadata.name}: {metadata.description}")print(f"Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

By referencing entities by their `@id`, we ensure clarity and reproducibility.

In [ ]:
# List all record sets and their fields using @id referencesrecord_sets = []if hasattr(metadata, 'recordSet') and metadata.recordSet:    record_sets = metadata.recordSet    print(f"Record Sets found ({len(record_sets)}):")    for rs in record_sets:        print(f"- Record Set @id: {rs['@id']}")        if 'field' in rs:            print("  Fields:")            for fld in rs['field']:                print(f"    - Field @id: {fld['@id']} (name: {fld.get('name', '<none>')})")else:    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract all available record sets into DataFrames, referenced by their `@id`.

In [ ]:
# Extract data from each record set using their @idsdataframes = {}record_set_ids = []fields_by_record_set = {}
# Gather all record set @idsif hasattr(metadata, 'recordSet') and metadata.recordSet:    for rs in metadata.recordSet:        record_set_ids.append(rs['@id'])        if 'field' in rs:            fields_by_record_set[rs['@id']] = [fld['@id'] for fld in rs['field']]
print('Record Sets (@id):', record_set_ids)
# Load each record set into a DataFramefor rs_id in record_set_ids:    records = list(dataset.records(record_set=rs_id))    df = pd.DataFrame(records)    dataframes[rs_id] = df    print(f"Loaded DataFrame for Record Set @id: {rs_id} with shape {df.shape}")
# Preview columns of the first loaded DataFrame (if exists)if record_set_ids:    first_rs_id = record_set_ids[0]    print(f"Columns in DataFrame ({first_rs_id}):")    print(dataframes[first_rs_id].columns.tolist())    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.

Reference all fields by their `@id`. Here, we:* Filter on a numeric field (e.g., GAD-7 total score)
* Normalize that column
* Group by a demographic attribute (e.g., gender)

_Adjust numeric_field and group_field variables as appropriate based on the actual dataset fields._

In [ ]:
# Example EDA using explicit @id referencing# Set correct record set and field @ids based on Data Overview step.
# Replace these with the actual @ids found above.

# Suppose the main survey record set is:# 'https://api.app.sen.science/frontiers/7160411/fair2-main-recordset' (example only!)# Check your output from section 2 and use actual values.

if record_set_ids:    record_set_id = record_set_ids[0]else:    record_set_id = None
# Example field @ids (replace as needed):numeric_field_id = Nonegroup_field_id = None
# Attempt to choose a numeric field from the columns for demonstration purposesdf = dataframes.get(record_set_id, pd.DataFrame())if not df.empty:    # Find first numeric column    for col in df.columns:        if pd.api.types.is_numeric_dtype(df[col]):            numeric_field_id = col
            break    print(f"Using numeric field: {numeric_field_id}")
    # Find first candidate for grouping (string categorical)    group_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]    if group_candidates:        group_field_id = group_candidates[0]
        print(f"Using group field: {group_field_id}")

    # EDA operations    threshold = df[numeric_field_id].mean() if numeric_field_id else 10    filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df.copy()    print(f"Filtered records with {numeric_field_id} > {threshold}:")    print(filtered_df.head())
    # Normalize numeric field    if numeric_field_id:
        norm_col = f"{numeric_field_id}_normalized"        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")        print(filtered_df[[numeric_field_id, norm_col]].head())
    # Group by group_field (e.g., gender) and compute stats    if group_field_id and group_field_id in filtered_df.columns and numeric_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()        print(f"Grouped data by {group_field_id}:")        print(grouped_df.head())else:    print("No data available for EDA.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

Below, we plot a histogram of a numeric field (referenced by its `@id`) and show the mean by group.

In [ ]:
import matplotlib.pyplot as pltimport seaborn as sns
# Visualize only if EDA step produced meaningful output
if not df.empty and numeric_field_id:    plt.figure(figsize=(8,4))    sns.histplot(df[numeric_field_id], bins=15, kde=True)    plt.title(f"Distribution of {numeric_field_id} (by @id)")    plt.xlabel(numeric_field_id)    plt.ylabel("Count")    plt.show()
    # Grouped mean visualization if available
    if group_field_id and group_field_id in df.columns:        plt.figure(figsize=(7,5))        group_means = df.groupby(group_field_id)[numeric_field_id].mean().reset_index()        sns.barplot(x=group_field_id, y=numeric_field_id, data=group_means)        plt.title(f"Mean {numeric_field_id} by {group_field_id} (both by @id)")        plt.xlabel(group_field_id)        plt.ylabel(f"Mean {numeric_field_id}")        plt.show()

## 6. Conclusion
This notebook demonstrated how to access, explore, and analyze the FAIR² Mental Health Survey dataset via its Croissant schema using the `mlcroissant` library.

**Summary of Steps:**
- All entities (record sets, fields, columns) referenced strictly by their `@id`
- Dataset metadata and records loaded dynamically
- Exploratory analysis and visualization performed

Further analysis can be customized by referencing additional `@id`s for demographic, symptom, or assessment fields.